In [10]:
"""
Notebook: model_base_v1.ipynb
Objetivo: Entrenamiento y evaluación de un modelo base de Logistic Regression,
con el fin de realizar validación cruzada y contrastar su rendimiento frente a
modelos avanzados, dado que presenta valores de ROC-AUC comparables.
Autor: Jesús Rodríguez
Fecha: 15/12/2025
"""

'\nNotebook: model_base_v1.ipynb\nObjetivo: Entrenamiento y evaluación de un modelo base de Logistic Regression, \ncon el fin de realizar validación cruzada y contrastar su rendimiento frente a\nmodelos avanzados, dado que presenta valores de ROC-AUC comparables.\nAutor: Jesús Rodríguez\nFecha: 15/12/2025\n'

## 1. Configuración inicial


In [27]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, FunctionTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    precision_recall_curve, auc, confusion_matrix, RocCurveDisplay
)

# Interpretabilidad
from IPython.display import display

# Google Drive
from google.colab import drive

In [12]:
# Montaje de Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Definición de ruta
DATA_PATH = '/content/drive/MyDrive/TFM/data/cleaned_dataset.csv'

# Carga del dataset
df = pd.read_csv(DATA_PATH, encoding='Latin-1')

## 2. División del dataset

In [14]:
# pylint: disable=C0103
# pylint: disable=W0621
# Las variables de entrenamiento/validación/test usan nombres clásicos de ML:
# X_train, y_train, X_val, X_test, etc.

# Target
TARGET = "DIABETE3"

# Separación de features y target
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# División: 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42,
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (154625, 22), Val: (51542, 22), Test: (51542, 22)


## 3. Pipeline de preprocesamiento

### 3.1. Separación por tipo de variable

In [19]:
# Variables nominales con varias categorías (para OneHot en RF)
CATEGORICAL_NOMINAL = ['BPHIGH4', '_RACE']

# Variables binarias (imputación 0/1 en determinista)
BINARY_VARS = ['BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2',
               'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK',
               'DIFFALON', 'DIFFDRES', 'SMOKE100', 'ADDEPEV2', 'SEX']

# Variables ordinales
CATEGORICAL_ORDINAL = ['GENHLTH', '_PACAT1', '_AGEG5YR', '_BMI5CAT']

# Variables numéricas
NUMERIC_VARS = ['EXEROFT1', '_FRUTSUM', '_VEGESUM']

### 3.2. Preprocesamiento determinista

In [20]:
def deterministic_preproc(X):
    """
    - Sustituye los valores codificados como -1 por NaN.
    - Convierte las variables binarias a formato 0/1,
      asignando 0 a los valores distintos de 1.
    """
    X = X.copy()

    # Reemplazo global de valores -1 por NaN
    X = X.replace(-1, np.nan)

    # Normalización de variables binarias
    for col in BINARY_VARS:
        X[col] = (X[col] == 1).astype(int)

    return X

### 3.3. Preprocesamiento de numéricas

In [21]:
def cap_outliers_numeric(X_input, numeric_vars):
    """
    Recorta valores extremos de variables numéricas al percentil 1 y 99.

    Args:
        X_input (pd.DataFrame): Dataset de entrada.
        numeric_vars (list): Lista de columnas numéricas a recortar.

    Returns:
        pd.DataFrame: Dataset con valores acotados.
    """
    X = X_input.copy()
    for col in numeric_vars:
        low = np.nanpercentile(X[col], 1)
        high = np.nanpercentile(X[col], 99)
        X[col] = X[col].clip(low, high)
    return X

### 3.4. Transformadores por tipo de variables

In [22]:
# Nominal: Most frequent y OneHot
nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ordinal: Most frequent
ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Numéricas: capping + median
numeric_pipeline = Pipeline(steps=[
    ('cap_outliers', FunctionTransformer(
        func=cap_outliers_numeric,
        kw_args={'numeric_vars': NUMERIC_VARS},
        validate=False
    )),
    ('imputer', SimpleImputer(strategy='median'))
])

### 3.5 ColumnTransformers

In [23]:
preprocessor = ColumnTransformer(transformers=[
    ('nom', nominal_pipeline, CATEGORICAL_NOMINAL),
    ('bin', 'passthrough', BINARY_VARS),
    ('ord', ordinal_pipeline, CATEGORICAL_ORDINAL),
    ('num', numeric_pipeline, NUMERIC_VARS),
])

## 4. Definición del modelo Logistic Regression

### 4.1. Modelo y pipeline final

In [24]:
lr_model = LogisticRegression(
    max_iter=1000,
    solver='liblinear',
    class_weight='balanced',
    random_state=42
    )


In [25]:
# Pipeline final
model_pipeline = {
    'Logistic Regression': Pipeline([
        ('deterministic', FunctionTransformer(deterministic_preproc)),
        ('preprocessor', preprocessor),
        ('model', lr_model)
    ])
}

### 4.2. Configuración de validación cruzada

In [28]:
# Estratificación para mantener proporción de clases
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a calcular
scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'       # PR-AUC, útil en dataset desbalanceado
}

### 4.3. Evaluación del modelo mediante cross-validation

In [30]:
cv_results = []

for model_name, pipeline in model_pipeline.items():
    # cross_validate devuelve dict con test_roc_auc y test_pr_auc
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        'Model': model_name,
        'ROC-AUC (CV mean)': scores['test_roc_auc'].mean(),
        'PR-AUC (CV mean)': scores['test_pr_auc'].mean(),
        'ROC-AUC (CV std)': scores['test_roc_auc'].std()
    })

# Resultados en DataFrame
df_cv = pd.DataFrame(cv_results)

print("=== VALIDACIÓN CRUZADA ===")
display(df_cv)

=== VALIDACIÓN CRUZADA ===


,Model,ROC-AUC (CV mean),PR-AUC (CV mean),ROC-AUC (CV std)
0,Logistic Regression,0.836189,0.062672,0.003062
